# Comparing SC weight parameterisations: Dirichlet vs softmax-Normal

CausalPy offers two model classes for Bayesian {term}`synthetic control` that both produce simplex-valued weights (positive, sum to one) but encode different prior beliefs:

- **Dirichlet** ({class}`~causalpy.pymc_models.WeightedSumFitter`): the concentration parameter `a` controls sparsity vs uniformity. With `a=1` the prior is uniform on the simplex.
- **Softmax-Normal** ({class}`~causalpy.pymc_models.SoftmaxWeightedSumFitter`): places Normal priors on unconstrained logits and maps them to the simplex via softmax. The prior scale `sigma` continuously interpolates between DiD-like uniform weights (small sigma) and SC-like sparse weights (large sigma).

The softmax-Normal parameterisation is the Bayesian analogue of the $\ell_2$ regularisation parameter $\zeta$ in the frequentist Synthetic Difference-in-Differences of {cite:t}`arkhangelsky2021synthetic`.

This notebook is a focused companion to {doc}`sc_pymc_brexit` — see that notebook for the full {term}`synthetic control` pipeline including donor pool selection, diagnostics, and the convex hull assumption.

In [ ]:
import arviz as az
import causalpy as cp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pymc_extras.prior import Prior

%config InlineBackend.figure_format = "retina"
seed = 123

## Setup

We use the Brexit GDP dataset with 15 donor countries. For details on donor pool selection, see {doc}`sc_pymc_brexit`.

In [ ]:
df = (
    cp.load_data("brexit")
    .assign(Time=lambda x: pd.to_datetime(x["Time"]))
    .set_index("Time")
    .loc[lambda x: x.index >= "2009-01-01"]
    .drop(["Japan", "Italy", "US", "Spain", "Portugal"], axis=1)
)

treatment_time = pd.to_datetime("2016 June 24")
target_country = "UK"
other_countries = [c for c in df.columns if c != target_country]

sample_kwargs = {
    "tune": 1000,
    "draws": 1000,
    "chains": 2,
    "random_seed": seed,
    "target_accept": 0.95,
}

print(f"Donors ({len(other_countries)}): {other_countries}")
df.head()